<a href="https://colab.research.google.com/github/Maulhdz/social-media-mental-health-clustering/blob/main/UAS_DMDW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Segmentasi dan Analisis Perilaku Digital Gen Z menggunakan K-Means Clustering: Pengaruh Intensitas Penggunaan Media Sosial terhadap Kesehatan Mental**

Tujuan penelitian ini adalah untuk melakukan analisis data mining
dalam mengidentifikasi segmentasi perilaku pengguna Gen Z berdasarkan
intensitas penggunaan media sosial dan menganalisis hubungannya dengan
kesehatan mental, dengan pertanyaan penelitian sebagai berikut:

1. Apakah pengguna Gen Z dapat disegmentasikan ke dalam kelompok-kelompok
   yang berbeda berdasarkan pola perilaku digital mereka menggunakan
   K-Means Clustering?

2. Apakah terdapat hubungan signifikan antara intensitas penggunaan media
   sosial dan kesehatan mental di setiap segment pengguna?

3. Apakah tingkat kesehatan mental berbeda secara signifikan antar segment
   pengguna berdasarkan intensitas penggunaan media sosial?


Tahapan Penelitian:
1. Data Loading & Inspection: Memahami struktur data dan mengidentifikasi kolom yang relevan untuk analisis.
2. Data Preparation:  Menghilangkan noise dan mempersiapkan data agar siap untuk dianalisis.
3. EDA (Descriptive Stats & Distribution): Menemukan pola awal, distribusi variabel, dan menentukan teknik statistik yang tepat di tahap berikutnya.
4. Feature Selection: Memilih fitur yang paling relevan untuk clustering dan menyamakan skala agar K-Means berjalan optimal.
5. K-Means Clustering: Mengelompokkan users menjadi 3 segment berdasarkan perilaku intensitas penggunaan media sosial mereka.
6. Cluster Profiling:Memahami karakteristik unik setiap cluster untuk melihat siapa saja users dalam masing-masing segment.
7. Within-Cluster Correlation Analysis: Menganalisis apakah hubungan intensitas dengan mental health konsisten atau berbeda di setiap cluster.
8. Cross-Cluster Statistical Comparison: Membuktikan secara statistik bahwa mental health score benar-benar berbeda signifikan antar cluster.
9. Inegration, interpretation, Visualization: Mengintegrasikan semua temuan menjadi insight yang actionable dan divisualkan dalam dashboard untuk menjawab research question.


## **Data Load & Inspection**
 merupakan langkah awal yang krusial dalam analisis data (Data Mining/EDA) yang bertujuan untuk memuat dataset external ke dalam lingkungan pemrograman menggunakan library pendukung (seperti pandas dan numpy), sekaligus melakukan pemeriksaan awal terhadap struktur fisik data. Melalui proses ini, Anda dapat "berkenalan" dengan properti data melalui pengecekan nama-nama kolom, peninjauan sampel baris teratas, hingga mengetahui dimensi total data yang tersedia. Pada proyek riset Anda ini, tahapan ini juga langsung dimanfaatkan untuk melakukan penyaringan awal (filtering) guna memisahkan subset data spesifik Gen Z (usia 12–27 tahun) agar analisis dan segmentasi menggunakan K-Means Clustering pada tahapan berikutnya menjadi lebih fokus dan akurat.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Load dataset
df = pd.read_csv('/content/social_media_user_behavior.csv')

# inspek kualitas data
# missing values
print("\n" + "=" * 50)
print("1. MISSING VALUES")
print("=" * 50)
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "Tidak ada missing value yang ditemukan!")

print("\n" + "=" * 50)
print("2. DUPLICATE DATA")
print("=" * 50)
dupes = df.duplicated().sum()
print(f"{'Tidak ada duplikat!' if dupes == 0 else f'ditemukan data duplikat sebanyak: {dupes}'}")

print("\n" + "=" * 20)
print("3. OUTLIER CHECK (Numerical Column - IQR metdhod)")
print("=" * 20)
num_cols = ['daily_usage_hours', 'sessions_per_day', 'avg_session_duration_min',
            'self_reported_mental_health_score', 'followers_count', 'following_count',
            'posts_per_week', 'likes_given_per_day', 'platforms_used_count']

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    pct = len(outliers) / len(df) * 100
    status = "⚠️" if len(outliers) > 0 else "✅"
    print(f"{status} {col}: {len(outliers)} outliers ({pct:.1f}%) | Range: [{lower:.1f}, {upper:.1f}]")

print("\n" + "=" * 50)
print("4. VALUE RANGE CHECK (Sanity Check)")
print("=" * 50)
for col in num_cols:
    print(f"{col}: min={df[col].min():.2f}, max={df[col].max():.2f}, mean={df[col].mean():.2f}")

print("\n" + "=" * 50)
print("5. CATEGORICAL CONSISTENCY")
print("=" * 50)
cat_cols = ['gender', 'sleep_disruption', 'mood_while_scrolling',
            'screen_time_concern', 'takes_social_media_breaks', 'primary_platform']
for col in cat_cols:
    print(f"{col}: {df[col].unique()}")



1. MISSING VALUES
Tidak ada missing value yang ditemukan!

2. DUPLICATE DATA
Tidak ada duplikat!

3. OUTLIER CHECK (Numerical Column - IQR metdhod)
⚠️ daily_usage_hours: 58 outliers (2.9%) | Range: [-1.8, 7.3]
⚠️ sessions_per_day: 25 outliers (1.2%) | Range: [-1.5, 10.5]
⚠️ avg_session_duration_min: 155 outliers (7.8%) | Range: [-36.6, 111.5]
⚠️ self_reported_mental_health_score: 3 outliers (0.1%) | Range: [1.2, 11.2]
⚠️ followers_count: 241 outliers (12.0%) | Range: [-2976.4, 5498.6]
⚠️ following_count: 172 outliers (8.6%) | Range: [-597.5, 1286.5]
⚠️ posts_per_week: 17 outliers (0.9%) | Range: [-1.0, 7.0]
⚠️ likes_given_per_day: 11 outliers (0.5%) | Range: [8.0, 32.0]
✅ platforms_used_count: 0 outliers (0.0%) | Range: [-1.0, 7.0]

4. VALUE RANGE CHECK (Sanity Check)
daily_usage_hours: min=0.50, max=12.00, mean=2.99
sessions_per_day: min=1.00, max=15.00, mean=5.02
avg_session_duration_min: min=3.30, max=594.00, mean=47.55
self_reported_mental_health_score: min=1.00, max=10.00, mean=6

**Hasil:**

Setelah dilakukan data inspeksi data, ditemukan:
* Dataset terdiri dari 2000 rows x 34 columns
* Data konsisten dan bersih dari nilai null serta duplikat
* Data memiliki banyak sekali  outlier yang perlu diatasi

**Variabel yang akan dibutuhkan untuk analsisis:**

Intensitas Penggunaan:

*   daily_usage_hours [float]
*   session_per_day [int]
*   avg_session_duration_min [float]
*   platform_used_count [int]
*   screen_time_concern [categorical: ya/tidak]

Kesehatan mental:
* self_reported_mental_health_score [float]
* sleep_disruption [categorical]
* mood_while_scrolling [Happy/Neutral/Inspired/etc]
* takes_social_media_breaks [Yes/No]


**Langkah Selanjutnya:**

Kami akan membersihkan data outlier, namun perlu  difilter terlebih dahulu data tersebut menjadi rentang umur genz, sehingga yang kami lakukan adalah dengan membuat dataframe baru dengan nama df_genz.

## **Filter data untuk Gen z**

Tahapan ini kami melakukan pemfilteran data sesuai dengan rentang usia yang menjadi target penelitian kami yaitu gen z, yaitu individu dalam rentang usia 12 sd 27. Hal ini untuk memastikan bahwa analisis selanjutnya, yaitu K-Means Clustering, dilakukan pada demografi target.

In [ ]:
# Filter for Gen Z users (age 12-27)
df_genz = df[(df['age'] >= 12) & (df['age'] <= 27)]

print("CONTOH DATA GENZ")
display(df_genz.head())

total_umum = len(df)
total_genz = len(df_genz)
persentase_genz = total_genz / total_umum * 100

print(f"Jumlah baris: {df_genz.shape[0]}, Jumlah kolom: {df_genz.shape[1]}")

print("\nPerbandingan data umum dengan data genz:")
print(f"Data umum: {total_umum}")
print(f"Data Gen Z: {total_genz}")
print(f"Persentase: {persentase_genz:.2f}% dari total data")

CONTOH DATA GENZ


,user_id,age,gender,country,profession,primary_platform,platforms_used_count,daily_usage_hours,sessions_per_day,avg_session_duration_min,...,primary_purpose,sleep_disruption,self_reported_mental_health_score,screen_time_concern,notification_frequency,privacy_setting,influencer_status,account_join_date,mood_while_scrolling,takes_social_media_breaks
1,USR00002,25,Male,USA,Entrepreneur,Instagram,2,2.4,5,28.8,...,News & Updates,Moderate impact,4.2,No,Selected,Public,No,2024-12-03,Neutral,Yes
4,USR00005,25,Male,UK,Other,YouTube,3,3.9,7,33.4,...,Entertainment,Mild impact,6.5,No,Do Not Disturb,Public,No,2023-09-08,Inspired,No
5,USR00006,25,Female,India,Other,Facebook,3,4.6,7,39.4,...,Business/Marketing,Mild impact,6.1,No,Selected,Friends Only,No,2023-07-18,Relaxed,Yes
8,USR00009,23,Female,Brazil,Marketer,Twitter/X,2,4.4,5,52.8,...,Learning,No impact,3.1,Yes,Selected,Friends Only,No,2024-05-24,Happy,Occasionally
10,USR00011,23,Female,USA,Freelancer,Instagram,3,3.9,7,33.4,...,News & Updates,No impact,1.0,Yes,Do Not Disturb,Public,No,2022-04-07,Happy,No


Jumlah baris: 1072, Jumlah kolom: 34

Perbandingan data umum dengan data genz:
Data umum: 2000
Data Gen Z: 1072
Persentase: 53.60% dari total data


### **Profiling Data Gen Z**

In [ ]:
# inspek oitlier data genz

print("\n" + "=" * 80)
print("OUTLIER CHECK (Numerical Column - IQR metdhod)")
print("=" * 80)
num_cols = ['daily_usage_hours', 'sessions_per_day', 'avg_session_duration_min',
            'self_reported_mental_health_score', 'followers_count', 'following_count',
            'posts_per_week', 'likes_given_per_day', 'platforms_used_count']

for col in num_cols:
    Q1 = df_genz[col].quantile(0.25)
    Q3 = df_genz[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df_genz[(df_genz[col] < lower) | (df_genz[col] > upper)]
    pct = len(outliers) / len(df_genz) * 100
    status = "⚠️" if len(outliers) > 0 else "✅"
    print(f"{status} {col}: {len(outliers)} outliers ({pct:.1f}%) | Range: [{lower:.1f}, {upper:.1f}]")

print("\n" + "=" * 50)
print("VALUE RANGE CHECK (Sanity Check)")
print("=" * 50)
for col in num_cols:
    print(f"{col}: min={df_genz[col].min():.2f}, max={df_genz[col].max():.2f}, mean={df_genz[col].mean():.2f}")



OUTLIER CHECK (Numerical Column - IQR metdhod)
⚠️ daily_usage_hours: 27 outliers (2.5%) | Range: [-2.1, 7.5]
⚠️ sessions_per_day: 16 outliers (1.5%) | Range: [-1.5, 10.5]
⚠️ avg_session_duration_min: 86 outliers (8.0%) | Range: [-36.0, 108.0]
⚠️ self_reported_mental_health_score: 4 outliers (0.4%) | Range: [1.3, 10.9]
⚠️ followers_count: 130 outliers (12.1%) | Range: [-2831.4, 5281.6]
⚠️ following_count: 92 outliers (8.6%) | Range: [-576.5, 1259.5]
⚠️ posts_per_week: 13 outliers (1.2%) | Range: [-1.0, 7.0]
⚠️ likes_given_per_day: 3 outliers (0.3%) | Range: [5.5, 33.5]
✅ platforms_used_count: 0 outliers (0.0%) | Range: [-1.0, 7.0]

VALUE RANGE CHECK (Sanity Check)
daily_usage_hours: min=0.50, max=12.00, mean=2.95
sessions_per_day: min=1.00, max=14.00, mean=5.07
avg_session_duration_min: min=3.30, max=594.00, mean=46.10
self_reported_mental_health_score: min=1.00, max=10.00, mean=6.15
followers_count: min=0.00, max=90336.00, mean=3058.62
following_count: min=6.00, max=10000.00, mean=526

**Hasil:**

Berdasarkan penelitian yang telah dilakukan, data genz memiliki outlier data yang signifikan, maka hal tersebut perlu diatasi agar tingkat akurasi klusterisasi menjadi tinggi

## **Data Cleaning & Preprocessing**

Tahapan ini berfokus pada penanganan pencilan (outlier) dengan strategi yang selektif. Berikut adalah poin penting dalam proses ini:

1.  **Fokus Fitur Utama:** Pembersihan outlier **hanya dilakukan pada fitur utama** yang akan digunakan untuk clustering (`daily_usage_hours`, `avg_session_duration_min`, `sessions_per_day`, `posts_per_week`, `likes_given_per_day`). Hal ini dilakukan agar model K-Means tidak terdistorsi oleh nilai ekstrem saat menghitung jarak antar kelompok.
2.  **Mempertahankan Data Strategis:** Kita sengaja **tidak menghapus outlier** pada kolom seperti `followers_count` atau `self_reported_mental_health_score`. Alasannya, nilai ekstrem pada jumlah pengikut adalah hal yang wajar (ada pengguna biasa vs influencer), dan skor kesehatan mental yang ekstrem justru merupakan data kunci yang ingin kita amati hubungannya dengan media sosial.
3.  **Filtrasi Gen Z:** Proses ini dilakukan setelah data difilter (usia 12-27 tahun) agar ambang batas IQR yang digunakan benar-benar merepresentasikan perilaku digital generasi ini, bukan populasi umum.

Dengan pendekatan ini, kita menjaga keseimbangan antara akurasi model clustering dan kekayaan informasi dataset.

In [ ]:
# 1. Membuat salinan data Gen Z
data_cleaned = df_genz.copy()
features = ['daily_usage_hours', 'avg_session_duration_min', 'sessions_per_day', 'posts_per_week', 'likes_given_per_day']

# Menghitung bounds BERDASARKAN data asli Gen Z (df_genz)
# Kita simpan dictionary berisi limits agar tidak berubah saat data dipotong
limits = {}
for col in features:
    Q1 = df_genz[col].quantile(0.25)
    Q3 = df_genz[col].quantile(0.75)
    IQR = Q3 - Q1
    limits[col] = (Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

# Filter data: Baris harus berada dalam limit yang sudah ditentukan di atas
mask = pd.Series(True, index=df_genz.index)
for col in features:
    lower, upper = limits[col]
    mask &= (df_genz[col] >= lower) & (df_genz[col] <= upper)

data_cleaned = df_genz[mask].copy()

total_bersih = len(data_cleaned)
data_dihapus = total_genz - total_bersih
persentase_pembersihan = data_dihapus / total_genz * 100

print(f"Jumlah data awal Gen Z: {df_genz.shape[0]} baris")
print(f"Jumlah data Gen Z setelah outlier DIHAPUS: {data_cleaned.shape[0]} baris")
print(f"Persentasi data yang berhasil dibersihkan: {persentase_pembersihan:.2f}% dari data genz")

Jumlah data awal Gen Z: 1072 baris
Jumlah data Gen Z setelah outlier DIHAPUS: 951 baris
Persentasi data yang berhasil dibersihkan: 11.29% dari data genz


In [ ]:
# Verifikasi ulang menggunakan limit asli dari df_genz
print("=== VERIFIKASI OUTLIER PADA DATA_CLEANED (MENGGUNAKAN LIMIT ASLI) ===")
for col in features:
    lower, upper = limits[col]
    outliers_remaining = data_cleaned[(data_cleaned[col] < lower) | (data_cleaned[col] > upper)]
    status = "✅" if len(outliers_remaining) == 0 else "⚠️"
    print(f"{status} {col}: {len(outliers_remaining)} outliers tersisa terhadap limit asli")

=== VERIFIKASI OUTLIER PADA DATA_CLEANED (MENGGUNAKAN LIMIT ASLI) ===
✅ daily_usage_hours: 0 outliers tersisa terhadap limit asli
✅ avg_session_duration_min: 0 outliers tersisa terhadap limit asli
✅ sessions_per_day: 0 outliers tersisa terhadap limit asli
✅ posts_per_week: 0 outliers tersisa terhadap limit asli
✅ likes_given_per_day: 0 outliers tersisa terhadap limit asli
